# z618 - Feature Engineering sobre escalado (Etapa C, consigna nueva)
Todo sobre `E_tn` (tn0 escalado). Historia: lags, delta lags, medias, tendencias, max-min. Categorias: sumatoria todos los productos, cat1/cat2/cat3/marca, mismo producto todos los clientes, mismo producto todos los tamanos.

**NO implementado (ambiguo en la consigna, no se reinterpreta):** "sumatoria muchos clientes" y "productos complementarios".

In [1]:
!pip install -q polars pyarrow

In [2]:
import os
import polars as pl
import warnings
warnings.filterwarnings("ignore")

In [3]:
PARAM = {
    'experimento': 'CP_FE01',
    'escalado_path': '/home/ds/exp/CP_ESC01/tb_escalado_CP.parquet',
    'productos_path': '/home/ds/datasets/tb_productos.txt',
    'lags': [1, 2, 3, 6, 12],
    'windows': [3, 6, 9, 12, 18, 24, 36]
}

ruta = os.path.join('/home/ds/exp', PARAM['experimento'])
os.makedirs(ruta, exist_ok=True)
print(ruta)

/home/ds/exp/CP_FE01


In [4]:
df = pl.read_parquet(PARAM['escalado_path'])
CLAVE = ["customer_id", "product_id"]
df = df.sort(CLAVE + ["periodo"])
print(df.shape)

(16648066, 14)


## Historia: lags de E_tn

In [5]:
lag_exprs = [
    pl.col("E_tn").shift(n).over(CLAVE).alias(f"E_tn_lag_{n}")
    for n in PARAM['lags']
]
df = df.with_columns(lag_exprs)

## Historia: delta entre lags consecutivos

In [6]:
lags_ordenados = sorted(PARAM['lags'])
delta_exprs = []
for a, b in zip(lags_ordenados[:-1], lags_ordenados[1:]):
    delta_exprs.append(
        (pl.col(f"E_tn_lag_{a}") - pl.col(f"E_tn_lag_{b}")).alias(f"E_tn_delta_lag_{a}_{b}")
    )
df = df.with_columns(delta_exprs)

## Historia: medias, max, min moviles de E_tn (shift(1) antes del rolling, nunca incluye el periodo actual)

In [7]:
df = df.with_columns(
    pl.col("E_tn").shift(1).over(CLAVE).alias("E_tn_shift1")
)

rolling_exprs = []
for w in PARAM['windows']:
    base = pl.col("E_tn_shift1")
    rolling_exprs += [
        base.rolling_mean(window_size=w, min_periods=1).over(CLAVE).alias(f"E_tn_media_{w}"),
        base.rolling_max(window_size=w, min_periods=1).over(CLAVE).alias(f"E_tn_max_{w}"),
        base.rolling_min(window_size=w, min_periods=1).over(CLAVE).alias(f"E_tn_min_{w}"),
    ]
df = df.with_columns(rolling_exprs)

## Historia: tendencia (media corta vs media larga, escalado)

In [8]:
windows_ordenadas = sorted(PARAM['windows'])
tendencia_exprs = []
for corta, larga in zip(windows_ordenadas[:-1], windows_ordenadas[1:]):
    tendencia_exprs.append(
        (pl.col(f"E_tn_media_{corta}") / (pl.col(f"E_tn_media_{larga}") + 1e-6)).alias(f"E_tn_tendencia_{corta}_{larga}")
    )
df = df.with_columns(tendencia_exprs)

## Categorias: funcion generica leave-one-out sobre E_tn

In [9]:
def ratio_leave_one_out(df, group_cols, metric, nombre):
    agg = df.group_by(group_cols + ["periodo"]).agg(
        pl.col(metric).sum().alias("_suma_grupo"),
        pl.len().alias("_n_grupo")
    )
    out = df.join(agg, on=group_cols + ["periodo"], how="left")
    out = out.with_columns(
        (
            (pl.col("_suma_grupo") - pl.col(metric))
            / (pl.col("_n_grupo") - 1).clip(lower_bound=1)
        ).alias(f"{metric}_prom_{nombre}_excl")
    )
    out = out.with_columns(
        (pl.col(metric) / (pl.col(f"{metric}_prom_{nombre}_excl") + 1e-6)).alias(
            f"ratio_{metric}_{nombre}"
        )
    )
    return out.drop(["_suma_grupo", "_n_grupo", f"{metric}_prom_{nombre}_excl"])

## Categorias: sumatoria TODOS los productos (global, leave-one-out)

In [10]:
df = ratio_leave_one_out(df, [], "E_tn", "macro")

## Categorias: cat1, cat2, cat3, marca (mi jerarquia)

In [11]:
tb_productos = pl.read_csv(PARAM['productos_path'], separator="\t").select(
    ["product_id", "cat1", "cat2", "cat3", "brand", "sku_size", "descripcion"]
)
df = df.join(tb_productos, on="product_id", how="left")

for grupo in [["cat1"], ["cat2"], ["cat3"], ["brand"]]:
    nombre = "_".join(grupo)
    df = ratio_leave_one_out(df, grupo, "E_tn", nombre)

## Categorias: mismo producto, TODOS los clientes (leave-one-out por product_id)

In [12]:
df = ratio_leave_one_out(df, ["product_id"], "E_tn", "prod_todos_cli")

## Categorias: mismo producto, TODOS los tamanos (por descripcion, leave-one-out)

In [13]:
df = ratio_leave_one_out(df, ["descripcion"], "E_tn", "todos_tamanos")

In [14]:
volumen_cliente = df.group_by("customer_id").agg(pl.col("tn0").sum().alias("_tn_total_cliente"))
volumen_cliente = volumen_cliente.with_columns(
    pl.col("_tn_total_cliente").qcut(5, labels=["q1", "q2", "q3", "q4", "q5"]).alias("quintil_cliente")
)
df = df.join(volumen_cliente.select(["customer_id", "quintil_cliente"]), on="customer_id", how="left")

df = ratio_leave_one_out(df, ["product_id", "quintil_cliente"], "E_tn", "muchos_cli")
df = df.drop("quintil_cliente")

In [15]:
familia = df.group_by(["cat3", "brand", "periodo"]).agg(
    pl.col("E_tn").sum().alias("_suma_familia"),
    pl.len().alias("_n_familia")
)
mi_variante = df.group_by(["cat3", "brand", "descripcion", "periodo"]).agg(
    pl.col("E_tn").sum().alias("_suma_mi_variante"),
    pl.len().alias("_n_mi_variante")
)

df = df.join(familia, on=["cat3", "brand", "periodo"], how="left")
df = df.join(mi_variante, on=["cat3", "brand", "descripcion", "periodo"], how="left")

df = df.with_columns([
    (pl.col("_suma_familia") - pl.col("_suma_mi_variante")).alias("_suma_complementarios"),
    (pl.col("_n_familia") - pl.col("_n_mi_variante")).clip(lower_bound=1).alias("_n_complementarios"),
])
df = df.with_columns(
    (pl.col("_suma_complementarios") / pl.col("_n_complementarios")).alias("_promedio_complementarios")
)
df = df.with_columns(
    (pl.col("E_tn") / (pl.col("_promedio_complementarios") + 1e-6)).alias("ratio_E_tn_complementarios")
)
df = df.drop(["_suma_familia", "_n_familia", "_suma_mi_variante", "_n_mi_variante",
              "_suma_complementarios", "_n_complementarios", "_promedio_complementarios"])

## Guardar

In [16]:
salida = os.path.join(ruta, "tb_FE_CP.parquet")
df.write_parquet(salida)
print(salida)
print(df.shape)

/home/ds/exp/CP_FE01/tb_FE_CP.parquet
(16648066, 66)
